# 03. 백테스트 기반 보정 에이전트

목표: Nexus의 Calibration Agent 아이디어를 작은 코드로 구현한다.

실행 방법:
1. 앞의 두 노트북을 먼저 읽는다.
2. 필요한 패키지가 없다면 `pip install -r requirements.txt`를 실행한다.
3. 이 노트북을 위에서 아래로 실행한다.
4. 보정 규칙이 훈련 폴드에서는 좋아 보여도 검증 폴드에서 통과하지 못할 수 있음을 확인한다.

핵심 질문: 과거 오차에서 배운 규칙을 언제 믿고, 언제 버려야 할까?

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(33)

## 1. 보정 실험용 데이터

이번 데이터에는 긍정 이벤트와 부정 이벤트가 여러 번 반복된다. 보정 에이전트는 과거 폴드에서 이벤트 반영이 과했는지 부족했는지 찾는다.

In [ ]:
def make_dataset(n=132):
    weeks = np.arange(1, n + 1)
    base = 120 + weeks * 0.35 + 5 * np.sin(2 * np.pi * weeks / 13)
    event_effect = np.zeros(n)
    events = []

    positive_windows = [(28, 33), (74, 80)]
    negative_windows = [(50, 57), (101, 108)]

    for i, week in enumerate(weeks):
        if any(start <= week <= end for start, end in positive_windows):
            event_effect[i] += 9
            events.append("positive catalyst")
        elif any(start <= week <= end for start, end in negative_windows):
            event_effect[i] -= 13
            events.append("negative demand shock")
        else:
            events.append("ordinary week")

    value = base + event_effect + rng.normal(0, 1.8, n)
    return pd.DataFrame({"week": weeks, "value": value, "event": events})


df = make_dataset()
df.head()

## 2. 보정 전 예측 함수

이 함수는 앞 노트북의 축소 Nexus를 더 간단히 만든 것이다. `event_weight`가 클수록 이벤트 신호를 강하게 반영한다. 보정 에이전트는 이 값을 조정하는 규칙을 학습한다.

In [ ]:
def event_signal(text):
    text = text.lower()
    if "positive" in text:
        return 1.0
    if "negative" in text or "shock" in text:
        return -1.0
    return 0.0


def predict_with_rule(history, future_frame, event_weight=7.0, drift_window=8):
    """간단한 거시 추세와 이벤트 보정을 결합한다.

    history: 예측 시점까지 관측 가능한 과거 데이터
    future_frame: 예측할 미래 시점의 week와 event 정보
    event_weight: 이벤트 신호를 얼마나 강하게 숫자에 반영할지 결정한다.
    """
    x = history["week"].to_numpy()
    y = history["value"].to_numpy()
    slope, intercept = np.polyfit(x, y, deg=1)
    macro = intercept + slope * future_frame["week"].to_numpy()

    recent_drift = float(history["value"].diff().tail(drift_window).mean())
    micro_anchor = float(history["value"].iloc[-1])

    preds = []
    for step, row in enumerate(future_frame.itertuples(index=False), start=1):
        decay = np.exp(-(step - 1) / 5)
        micro_anchor = micro_anchor + recent_drift + event_signal(row.event) * event_weight * decay
        # 거시와 미시를 고정 비율로 결합한다. 보정은 event_weight를 바꿔 일어난다.
        preds.append(0.52 * macro[step - 1] + 0.48 * micro_anchor)
    return np.array(preds)


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))

## 3. 순차 백테스트 폴드 만들기

시계열에서는 과거에서 미래로 걸어가며 검증해야 한다. 각 폴드는 `history -> horizon` 구조를 갖는다.

In [ ]:
def make_backtest_folds(frame, start=52, horizon=8, step=12, max_folds=6):
    folds = []
    cutoff = start
    while cutoff + horizon <= len(frame) and len(folds) < max_folds:
        history = frame.iloc[:cutoff].copy()
        future = frame.iloc[cutoff:cutoff + horizon].copy()
        folds.append({"cutoff": cutoff, "history": history, "future": future})
        cutoff += step
    return folds


folds = make_backtest_folds(df)
[(fold["cutoff"], int(fold["future"].week.min()), int(fold["future"].week.max())) for fold in folds]

## 4. Calibration Agent: 오류에서 규칙 후보 만들기

논문에서는 LLM이 예측 오차와 추론을 분석해 지침을 만든다. 여기서는 단순하게 이벤트가 있는 구간에서 예측이 지속적으로 낮거나 높으면 규칙 후보를 만든다.

In [ ]:
def propose_rules_from_fold(fold, event_weight=7.0):
    future = fold["future"]
    pred = predict_with_rule(fold["history"], future, event_weight=event_weight)
    actual = future["value"].to_numpy()
    error = actual - pred

    rows = future.copy()
    rows["error"] = error
    rows["signal"] = rows["event"].map(event_signal)

    rules = set()
    positive_errors = rows.loc[rows["signal"] > 0, "error"]
    negative_errors = rows.loc[rows["signal"] < 0, "error"]

    # 실제값이 예측보다 높다면 긍정 이벤트 효과를 과소반영한 것이다.
    if len(positive_errors) >= 2 and positive_errors.mean() > 2.0:
        rules.add("increase_positive_event_weight")

    # 실제값이 예측보다 낮다면 부정 이벤트 효과를 과소반영한 것이다.
    if len(negative_errors) >= 2 and negative_errors.mean() < -2.0:
        rules.add("increase_negative_event_weight")

    # 이벤트가 없는 구간에서 예측이 계속 높으면 드리프트가 과한 것으로 본다.
    neutral_errors = rows.loc[rows["signal"] == 0, "error"]
    if len(neutral_errors) >= 4 and neutral_errors.mean() < -2.0:
        rules.add("dampen_recent_drift")

    return rules, mape(actual, pred)


train_folds = folds[:-1]
validation_fold = folds[-1]

rule_sets = []
for fold in train_folds:
    rules, score = propose_rules_from_fold(fold)
    rule_sets.append(rules)
    print(f"cutoff={fold['cutoff']}, MAPE={score:.4f}, rules={sorted(rules)}")

## 5. 반복적으로 나타난 규칙만 남기기

논문은 여러 훈련 폴드에서 공통으로 나타나는 지침을 남긴다. 완전한 교집합은 너무 엄격할 수 있으므로, 실습에서는 절반 이상 폴드에서 나온 규칙을 안정 규칙으로 본다.

In [ ]:
def stable_rules(rule_sets, min_fraction=0.5):
    counts = {}
    for rules in rule_sets:
        for rule in rules:
            counts[rule] = counts.get(rule, 0) + 1
    threshold = math.ceil(len(rule_sets) * min_fraction)
    return {rule for rule, count in counts.items() if count >= threshold}, counts


rules, counts = stable_rules(rule_sets)
rules, counts

## 6. 규칙을 적용한 예측 함수

규칙은 모델 파라미터를 직접 학습한다기보다 예측 전략을 조정하는 지침으로 작동한다. 여기서는 이벤트 가중치와 드리프트 창을 바꾼다.

In [ ]:
def apply_rules(base_event_weight, base_drift_window, rules):
    event_weight = base_event_weight
    drift_window = base_drift_window

    if "increase_positive_event_weight" in rules:
        event_weight += 1.5
    if "increase_negative_event_weight" in rules:
        event_weight += 2.0
    if "dampen_recent_drift" in rules:
        drift_window += 4

    return event_weight, drift_window


calibrated_event_weight, calibrated_drift_window = apply_rules(7.0, 8, rules)
calibrated_event_weight, calibrated_drift_window

## 7. 숨겨둔 검증 폴드에서 통과 여부 확인

훈련 폴드에서 만들어진 규칙은 바로 최종 테스트에 쓰지 않는다. 검증 폴드에서 최소 개선율을 넘을 때만 채택한다. 이 장치가 과적합을 줄인다.

In [ ]:
def evaluate_fold(fold, event_weight=7.0, drift_window=8):
    pred = predict_with_rule(fold["history"], fold["future"], event_weight=event_weight, drift_window=drift_window)
    actual = fold["future"]["value"].to_numpy()
    return mape(actual, pred), pred


base_score, base_pred = evaluate_fold(validation_fold)
cal_score, cal_pred = evaluate_fold(
    validation_fold,
    event_weight=calibrated_event_weight,
    drift_window=calibrated_drift_window,
)

improvement = (base_score - cal_score) / base_score
accepted = improvement >= 0.05

pd.DataFrame([
    {"setting": "base", "MAPE": base_score},
    {"setting": "calibrated", "MAPE": cal_score},
    {"setting": "relative_improvement", "MAPE": improvement},
])

In [ ]:
print("accepted:", accepted)
print("rules:", sorted(rules))

plot_df = validation_fold["future"][["week", "value"]].copy()
plot_df["base"] = base_pred
plot_df["calibrated"] = cal_pred

ax = plot_df.plot(x="week", y=["value", "base", "calibrated"], figsize=(10, 4))
ax.set_title("Validation fold: base vs calibrated")
ax.set_ylabel("value")
plt.show()

## 정리

- 보정은 과거 오차를 설명 가능한 규칙으로 바꾸는 단계다.
- 규칙은 여러 훈련 폴드에서 반복될수록 신뢰도가 높다.
- 검증 폴드에서 최소 개선율을 통과해야 최종 적용할 수 있다.
- 이 실습의 규칙은 단순하지만, Nexus의 보정 에이전트가 왜 필요한지 보여준다.
- 실제 시스템에서는 규칙 생성, 검증, 폐기 기준을 로그로 남겨야 운영 중 설명 가능성이 유지된다.